In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from sklearn.model_selection import train_test_split

/home/e11935050/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-11 14:35:38.919325: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-11 14:35:38.919370: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-11 14:35:38.920792: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-11 14:35:38.929145: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is op

In [2]:
triplets_df_path = "../data/TRIPLETS_final_linked.csv"
df = pd.read_csv(triplets_df_path)
df = df[["entity1", "rel_type", "entity2"]]
df = df.drop_duplicates(ignore_index=True)
df = df.dropna()

In [3]:
# Map entities to IDs
entities = pd.concat([df['entity1'], df['entity2']]).unique()
entity2id = {e: i for i, e in enumerate(entities)}

# Map relations to IDs
relations = df['rel_type'].unique()
relation2id = {r: i for i, r in enumerate(relations)}

df['head_id'] = df['entity1'].map(entity2id)
df['tail_id'] = df['entity2'].map(entity2id)
df['rel_id'] = df['rel_type'].map(relation2id)

df = df.dropna(subset=['head_id','tail_id','rel_id'])
df[['head_id','tail_id','rel_id']] = df[['head_id','tail_id','rel_id']].astype(int)

In [4]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, shuffle=True
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

num_nodes = len(entity2id)
num_relations = len(relation2id)  # original relations


Train: 61254, Val: 7657, Test: 7657


In [5]:
def build_edge_index(df_subset):
    edge_index = torch.tensor([
        df_subset['head_id'].values,
        df_subset['tail_id'].values
    ], dtype=torch.long)
    edge_type = torch.tensor(df_subset['rel_id'].values, dtype=torch.long)

    # add inverse edges
    inv_edge_index = torch.stack([edge_index[1], edge_index[0]], dim=0)
    inv_edge_type = edge_type + num_relations

    edge_index = torch.cat([edge_index, inv_edge_index], dim=1)
    edge_type = torch.cat([edge_type, inv_edge_type], dim=0)
    return edge_index, edge_type

# Graph used for message passing (train only)
edge_index, edge_type = build_edge_index(train_df)

# Positive edges for train / val / test (without inverse)
train_pos_edge_index = torch.tensor([
    train_df['head_id'].values,
    train_df['tail_id'].values
], dtype=torch.long)
train_pos_edge_type = torch.tensor(train_df['rel_id'].values, dtype=torch.long)

val_pos_edge_index = torch.tensor([
    val_df['head_id'].values,
    val_df['tail_id'].values
], dtype=torch.long)
val_pos_edge_type = torch.tensor(val_df['rel_id'].values, dtype=torch.long)

test_pos_edge_index = torch.tensor([
    test_df['head_id'].values,
    test_df['tail_id'].values
], dtype=torch.long)
test_pos_edge_type = torch.tensor(test_df['rel_id'].values, dtype=torch.long)

/tmp/ipykernel_1229997/2347881502.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([


In [6]:
def negative_sampling(pos_edge_index, pos_edge_type, num_nodes):
    num_pos = pos_edge_index.size(1)
    corrupt_head = torch.rand(num_pos) < 0.5

    neg_heads = pos_edge_index[0].clone()
    neg_tails = pos_edge_index[1].clone()
    random_nodes = torch.randint(0, num_nodes, (num_pos,))

    neg_heads[corrupt_head] = random_nodes[corrupt_head]
    neg_tails[~corrupt_head] = random_nodes[~corrupt_head]

    neg_edge_index = torch.stack([neg_heads, neg_tails], dim=0)
    neg_edge_type = pos_edge_type.clone()
    return neg_edge_index, neg_edge_type

In [7]:
class RGCNLinkPredictor(nn.Module):
    def __init__(
        self,
        num_nodes,
        num_relations,
        hidden_channels=128,
        embedding_dim=128,
        num_bases=8,
        dropout=0.2
    ):
        super().__init__()
        self.entity_emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)

        self.conv1 = RGCNConv(embedding_dim, hidden_channels, num_relations*2, num_bases=num_bases)
        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations*2, num_bases=num_bases)

        self.rel_emb = nn.Embedding(num_relations*2, hidden_channels)
        nn.init.xavier_uniform_(self.rel_emb.weight)

        self.dropout = nn.Dropout(dropout)

    def encode(self, edge_index, edge_type):
        x = self.entity_emb.weight
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index, edge_type)
        return x

    def decode(self, edge_index, node_repr, edge_type):
        h = node_repr[edge_index[0]]
        t = node_repr[edge_index[1]]
        r = self.rel_emb(edge_type)
        return (h * r * t).sum(dim=1)  # DistMult

    def forward(self, edge_index, edge_type, pos_edge_index, pos_edge_type, neg_edge_index, neg_edge_type):
        node_repr = self.encode(edge_index, edge_type)
        pos_score = self.decode(pos_edge_index, node_repr, pos_edge_type)
        neg_score = self.decode(neg_edge_index, node_repr, neg_edge_type)
        return pos_score, neg_score


In [8]:
def evaluate_link_prediction_metrics(pos_score, neg_score, ks=[1,3,10]):
    pos_score = pos_score.cpu()
    neg_score = neg_score.cpu()
    num_pos = pos_score.shape[0]
    num_neg = neg_score.shape[0] // num_pos
    neg_score = neg_score.view(num_pos, num_neg)

    ranks = []
    hits_at_k = {k:0 for k in ks}

    for i in range(num_pos):
        scores = torch.cat([pos_score[i].unsqueeze(0), neg_score[i]])
        sorted_scores, indices = torch.sort(scores, descending=True)
        rank = (indices==0).nonzero(as_tuple=False).item()+1
        ranks.append(rank)
        for k in ks:
            if rank <= k:
                hits_at_k[k] += 1

    ranks_tensor = torch.tensor(ranks, dtype=torch.float)
    mrr = torch.mean(1.0/ranks_tensor).item()
    mr = torch.mean(ranks_tensor).item()
    hits = {k: hits_at_k[k]/num_pos for k in ks}
    return {"MRR": mrr, "MR": mr, "Hits": hits}


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RGCNLinkPredictor(num_nodes, num_relations).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    neg_edge_index, neg_edge_type = negative_sampling(train_pos_edge_index, train_pos_edge_type, num_nodes)

    pos_score, neg_score = model(
        edge_index.to(device),
        edge_type.to(device),
        train_pos_edge_index.to(device),
        train_pos_edge_type.to(device),
        neg_edge_index.to(device),
        neg_edge_type.to(device)
    )

    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)]).to(device)

    loss = criterion(scores, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.4f}")

Epoch 5/50 | Loss: 0.6452
Epoch 10/50 | Loss: 0.6012
Epoch 15/50 | Loss: 0.5423
Epoch 20/50 | Loss: 0.4319
Epoch 25/50 | Loss: 0.3346
Epoch 30/50 | Loss: 0.2798
Epoch 35/50 | Loss: 0.2344
Epoch 40/50 | Loss: 0.1996
Epoch 45/50 | Loss: 0.1745
Epoch 50/50 | Loss: 0.1539


In [10]:
model.eval()
with torch.no_grad():
    val_neg_edge_index, val_neg_edge_type = negative_sampling(val_pos_edge_index, val_pos_edge_type, num_nodes)
    pos_score, neg_score = model(
        edge_index.to(device),
        edge_type.to(device),
        val_pos_edge_index.to(device),
        val_pos_edge_type.to(device),
        val_neg_edge_index.to(device),
        val_neg_edge_type.to(device)
    )
    metrics = evaluate_link_prediction_metrics(pos_score, neg_score)
    print("Validation metrics:", metrics)

model.eval()
with torch.no_grad():
    test_neg_edge_index, test_neg_edge_type = negative_sampling(test_pos_edge_index, test_pos_edge_type, num_nodes)
    pos_score, neg_score = model(
        edge_index.to(device),
        edge_type.to(device),
        test_pos_edge_index.to(device),
        test_pos_edge_type.to(device),
        test_neg_edge_index.to(device),
        test_neg_edge_type.to(device)
    )
    metrics = evaluate_link_prediction_metrics(pos_score, neg_score)
    print("Test metrics:", metrics)

Validation metrics: {'MRR': 0.8555570244789124, 'MR': 1.2888859510421753, 'Hits': {1: 0.7111140133211441, 3: 1.0, 10: 1.0}}
Test metrics: {'MRR': 0.8594096899032593, 'MR': 1.2811806201934814, 'Hits': {1: 0.7188193809586, 3: 1.0, 10: 1.0}}


In [11]:
# Build a set of all true triples in the KG for filtering
all_true_triples = set(
    zip(
        df['head_id'].values,
        df['rel_id'].values,
        df['tail_id'].values
    )
)

def evaluate_filtered_metrics(pos_edge_index, pos_edge_type, num_nodes, model, edge_index, edge_type, all_true_triples, ks=[1,3,10]):
    """
    pos_edge_index, pos_edge_type: positive triples to evaluate
    num_nodes: total number of entities
    model: trained RGCN model
    edge_index, edge_type: message passing graph
    all_true_triples: set of (h,r,t) that exist in KG
    """
    model.eval()
    with torch.no_grad():
        node_repr = model.encode(edge_index, edge_type)  # full embedding

        num_pos = pos_edge_index.shape[1]
        ranks = []
        hits_at_k = {k:0 for k in ks}

        for i in range(num_pos):
            h = pos_edge_index[0, i].item()
            t = pos_edge_index[1, i].item()
            r = pos_edge_type[i].item()

            # scores for replacing tail
            tail_candidates = torch.arange(num_nodes)
            head_tensor = torch.tensor([h]*num_nodes)
            rel_tensor = torch.tensor([r]*num_nodes)
            candidate_edge_index = torch.stack([head_tensor, tail_candidates])
            candidate_edge_type = rel_tensor

            scores = model.decode(candidate_edge_index.to(node_repr.device),
                                  node_repr, candidate_edge_type.to(node_repr.device))

            # filter: remove all true triples except the current one
            mask = torch.ones(num_nodes, dtype=torch.bool)
            for c_t in range(num_nodes):
                if (h, r, c_t) in all_true_triples and c_t != t:
                    mask[c_t] = False
            filtered_scores = scores[mask]

            # compute rank of true tail (1-based)
            true_score = scores[t].item()
            rank = (filtered_scores >= true_score).sum().item()
            ranks.append(rank)

            for k in ks:
                if rank <= k:
                    hits_at_k[k] += 1

        ranks_tensor = torch.tensor(ranks, dtype=torch.float)
        mrr = torch.mean(1.0/ranks_tensor).item()
        mr = torch.mean(ranks_tensor).item()
        hits = {k: hits_at_k[k]/num_pos for k in ks}

        return {"MRR": mrr, "MR": mr, "Hits": hits}

filtered_metrics = evaluate_filtered_metrics(
    test_pos_edge_index,
    test_pos_edge_type,
    num_nodes,
    model,
    edge_index.to(model.entity_emb.weight.device),
    edge_type.to(model.entity_emb.weight.device),
    all_true_triples,
    ks=[1,3,10]
)

print("Filtered Test metrics:", filtered_metrics)

Filtered Test metrics: {'MRR': 0.00610653031617403, 'MR': 11029.5419921875, 'Hits': {1: 0.0009141961603761264, 3: 0.002742588481128379, 10: 0.01279874624526577}}


In [12]:
# Get the Invests_In relation ID
invests_in_rel_id = relation2id['invests_in']

# Filter test set for Invests_In
invest_test_mask = test_pos_edge_type == invests_in_rel_id
test_pos_edge_index_invest = test_pos_edge_index[:, invest_test_mask]
test_pos_edge_type_invest = test_pos_edge_type[invest_test_mask]

# ---------------------------------------------------------
# Filtered evaluation for Invests_In only
# ---------------------------------------------------------
def evaluate_filtered_invests_in(pos_edge_index, pos_edge_type, num_nodes, model, edge_index, edge_type, all_true_triples, ks=[1,3,10]):
    """
    Filtered metrics only for invests_in relation
    """
    model.eval()
    with torch.no_grad():
        node_repr = model.encode(edge_index, edge_type)
        num_pos = pos_edge_index.shape[1]
        ranks = []
        hits_at_k = {k:0 for k in ks}

        for i in range(num_pos):
            h = pos_edge_index[0, i].item()
            t = pos_edge_index[1, i].item()
            r = pos_edge_type[i].item()

            # Only evaluate Invests_In relation
            if r != invests_in_rel_id:
                continue

            # Candidate tail replacement
            tail_candidates = torch.arange(num_nodes)
            head_tensor = torch.tensor([h]*num_nodes)
            rel_tensor = torch.tensor([r]*num_nodes)
            candidate_edge_index = torch.stack([head_tensor, tail_candidates])
            candidate_edge_type = rel_tensor

            scores = model.decode(candidate_edge_index.to(node_repr.device),
                                  node_repr, candidate_edge_type.to(node_repr.device))

            # Filter: remove all true triples except current one
            mask = torch.ones(num_nodes, dtype=torch.bool)
            for c_t in range(num_nodes):
                if (h, r, c_t) in all_true_triples and c_t != t:
                    mask[c_t] = False
            filtered_scores = scores[mask]

            # Rank of true tail
            true_score = scores[t].item()
            rank = (filtered_scores >= true_score).sum().item()
            ranks.append(rank)

            for k in ks:
                if rank <= k:
                    hits_at_k[k] += 1

        ranks_tensor = torch.tensor(ranks, dtype=torch.float)
        mrr = torch.mean(1.0/ranks_tensor).item()
        mr = torch.mean(ranks_tensor).item()
        hits = {k: hits_at_k[k]/len(ranks) for k in ks}  # normalize by filtered triple count

        return {"MRR": mrr, "MR": mr, "Hits": hits}

## invest focus

In [13]:
filtered_invest_metrics = evaluate_filtered_invests_in(
    test_pos_edge_index_invest,
    test_pos_edge_type_invest,
    num_nodes,
    model,
    edge_index.to(model.entity_emb.weight.device),
    edge_type.to(model.entity_emb.weight.device),
    all_true_triples,
    ks=[1,3,10]
)

print("Filtered invests_in Test metrics:", filtered_invest_metrics)

Filtered invests_in Test metrics: {'MRR': 0.009692065417766571, 'MR': 11498.189453125, 'Hits': {1: 0.0, 3: 0.006535947712418301, 10: 0.026143790849673203}}


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv

# Assuming df has columns: head_id, tail_id, rel_id
num_nodes = len(entity2id)
num_relations = len(relation2id)
invest_rel_id = relation2id['invests_in']

# Build full message passing graph with inverse edges
edge_index = torch.tensor([
    df['head_id'].values,
    df['tail_id'].values
], dtype=torch.long)
edge_type = torch.tensor(df['rel_id'].values, dtype=torch.long)

# add inverse edges
inv_edge_index = torch.stack([edge_index[1], edge_index[0]], dim=0)
inv_edge_type = edge_type + num_relations
edge_index = torch.cat([edge_index, inv_edge_index], dim=1)
edge_type = torch.cat([edge_type, inv_edge_type], dim=0)

# Split into train/val/test with pykeen style
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

# Positive edges for training
train_pos_edge_index = torch.tensor([train_df['head_id'].values, train_df['tail_id'].values], dtype=torch.long)
train_pos_edge_type = torch.tensor(train_df['rel_id'].values, dtype=torch.long)

# Positive edges for validation/test
val_pos_edge_index = torch.tensor([val_df['head_id'].values, val_df['tail_id'].values], dtype=torch.long)
val_pos_edge_type = torch.tensor(val_df['rel_id'].values, dtype=torch.long)

test_pos_edge_index = torch.tensor([test_df['head_id'].values, test_df['tail_id'].values], dtype=torch.long)
test_pos_edge_type = torch.tensor(test_df['rel_id'].values, dtype=torch.long)

# All true triples for filtered evaluation
all_true_triples = set(zip(df['head_id'], df['rel_id'], df['tail_id']))

class RGCNLinkPredictor(nn.Module):
    def __init__(self, num_nodes, hidden_channels=64, num_relations=13, embedding_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        self.conv1 = RGCNConv(embedding_dim, hidden_channels, num_relations*2)  # include inverse relations
        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations*2)
        self.rel_embedding = nn.Embedding(num_relations*2, hidden_channels)

    def encode(self, edge_index, edge_type):
        x = self.embedding.weight
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_type)
        return x

    def decode(self, edge_index, x, edge_type):
        head, tail = edge_index
        head_emb = x[head]
        tail_emb = x[tail]
        rel_emb = self.rel_embedding(edge_type)
        score = (head_emb * rel_emb * tail_emb).sum(dim=1)
        return score

    def forward(self, edge_index, edge_type, pos_edge_index, pos_edge_type, neg_edge_index, neg_edge_type):
        x = self.encode(edge_index, edge_type)
        pos_score = self.decode(pos_edge_index, x, pos_edge_type)
        neg_score = self.decode(neg_edge_index, x, neg_edge_type)
        return pos_score, neg_score

def negative_sampling(pos_edge_index, pos_edge_type, num_nodes):
    num_neg = pos_edge_index.shape[1]
    neg_heads = pos_edge_index[0]
    neg_tails = torch.randint(0, num_nodes, (num_neg,))
    neg_edge_index = torch.stack([neg_heads, neg_tails], dim=0)
    neg_edge_type = pos_edge_type
    return neg_edge_index, neg_edge_type


# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = "cpu"
model = RGCNLinkPredictor(num_nodes, hidden_channels=64, num_relations=num_relations).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(reduction='none')  # we will apply weighting manually


num_epochs = 50
edge_index = edge_index.to(device)
edge_type = edge_type.to(device)
train_pos_edge_index = train_pos_edge_index.to(device)
train_pos_edge_type = train_pos_edge_type.to(device)

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    # negative sampling
    neg_edge_index, neg_edge_type = negative_sampling(train_pos_edge_index, train_pos_edge_type, num_nodes)
    neg_edge_index, neg_edge_type = neg_edge_index.to(device), neg_edge_type.to(device)

    # forward
    pos_score, neg_score = model(edge_index, edge_type, train_pos_edge_index, train_pos_edge_type,
                                 neg_edge_index, neg_edge_type)

    # BCE + weighting for Invests_In
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)]).to(device)
    weights = torch.ones_like(labels)
    # increase weight for Invests_In positives
    invest_mask = torch.cat([train_pos_edge_type == invest_rel_id,
                             neg_edge_type[:neg_score.shape[0]] == invest_rel_id])  # only pos part matters
    weights[invest_mask] = 5.0

    loss = (criterion(scores, labels) * weights).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.4f}")


filtered_invest_metrics = evaluate_filtered_invests_in(
    test_pos_edge_index,
    test_pos_edge_type,
    num_nodes,
    model,
    edge_index,
    edge_type,
    all_true_triples,
    ks=[1,3,10]
)

print("Filtered Invests_In Test metrics:", filtered_invest_metrics)

Epoch 5/50 | Loss: 10.5090
Epoch 10/50 | Loss: 3.5390
Epoch 15/50 | Loss: 1.4530
Epoch 20/50 | Loss: 0.7521
Epoch 25/50 | Loss: 0.4911
Epoch 30/50 | Loss: 0.3542
Epoch 35/50 | Loss: 0.2686
Epoch 40/50 | Loss: 0.2279
Epoch 45/50 | Loss: 0.1949
Epoch 50/50 | Loss: 0.1714
Filtered Invests_In Test metrics: {'MRR': 0.011264986358582973, 'MR': 2688.601318359375, 'Hits': {1: 0.0, 3: 0.006756756756756757, 10: 0.016891891891891893}}
